In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from helpers import box_plot

df = pd.read_csv("../data/clean-survey.csv")

# Convert the frequency-based responses into ordered numeric values so they can be summarised consistently.
frequency_scale = {"Rarely": 1, "Sometimes": 2, "Often": 3, "Always": 4}
df_problem = df.copy()

frequency_problem_cols = [
    "info_visibility_diff",
    "feedback_not_useful_frequency",
    "stop_checking_data_frequency",
]

for col in frequency_problem_cols:
    df_problem[f"{col}_num"] = df_problem[col].map(frequency_scale)


### Problem Space Analysis

This section validates whether the problem space is relevant and significant for our participants. We do this in three stages:
1. Summarise agreement-based pain points using the original 0-4 Likert items.
2. Summarise frequency-based pain points by mapping `Rarely` to `Always` onto an ordered 1-4 scale.
3. Compare current wearable users with former or non-current users to understand which frictions may be connected to disengagement.

Together, these analyses help us answer two questions: how widespread the problem is, and how severe participants perceive it to be.

#### 1. Agreement-Based Pain Points

We begin with the survey items already measured on a `0-4` agreement scale, where `0 = Strongly Disagree` and `4 = Strongly Agree`. These items capture whether participants experience key wearable-related frustrations.

In [ ]:
likert_problem_mapping = {
    "Information Overload": "information_overload",
    "Interpretation Difficulty": "data_interpretation_difficulty",
    "Frustration": "frustrated_feeling",
    "Battery Anxiety": "battery_anxiety",
    "Actionable Feedback Gap": "actionable_insight_gap",
}

likert_problem_df = df_problem[list(likert_problem_mapping.values())].rename(
    columns={value: key for key, value in likert_problem_mapping.items()}
)

likert_summary = likert_problem_df.aggregate(["mean", "median", "std"]).transpose().round(2)
display(likert_summary)

box_plot(likert_problem_df, "Agreement-Based Problem Indicators", "Likert Scale (0-4)", False)

agreement_share = (
    likert_problem_df.ge(3).sum().div(len(likert_problem_df)).mul(100).round(1)
).to_frame("Agree or Strongly Agree (%)").sort_values(
    by="Agree or Strongly Agree (%)", ascending=False
)
display(agreement_share)

ax = agreement_share.plot(kind="bar", legend=False, figsize=(8, 4), color="#3d7ea6")
ax.set_title("Share of Participants Reporting Stronger Problem Agreement")
ax.set_ylabel("Participants (%)")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


The strongest agreement-based issues are **battery anxiety** and the **actionable feedback gap**, with both endorsed by `44.4%` of participants at the `Agree` or `Strongly Agree` level. **Information overload** is also notable, while **frustration** and **data interpretation difficulty** appear somewhat lower on average.

This suggests the problem is not simply that users dislike wearables overall. Instead, the more convincing pattern is that users often feel the information they receive is not useful enough to act on, and the device experience can be mentally demanding or inconvenient over time.

#### 2. Frequency-Based Frictions

Some Step 3 questions use frequency language rather than agreement language. To analyse them consistently, we map `Rarely = 1`, `Sometimes = 2`, `Often = 3`, and `Always = 4`. This lets us describe how repeatedly participants encounter these issues.

In [ ]:
frequency_problem_mapping = {
    "Visibility Difficulties": "info_visibility_diff_num",
    "Feedback Not Useful": "feedback_not_useful_frequency_num",
    "Stopped Checking Data": "stop_checking_data_frequency_num",
}

frequency_problem_df = df_problem[list(frequency_problem_mapping.values())].rename(
    columns={value: key for key, value in frequency_problem_mapping.items()}
)

frequency_summary = frequency_problem_df.aggregate(["mean", "median", "std"]).transpose().round(2)
display(frequency_summary)

box_plot(frequency_problem_df, "Frequency-Based Problem Indicators", "Frequency Scale (1-4)", False)

frequency_share = (
    frequency_problem_df.ge(2).sum().div(len(frequency_problem_df)).mul(100).round(1)
).to_frame("Sometimes or More (%)").sort_values(
    by="Sometimes or More (%)", ascending=False
)
display(frequency_share)

frequency_distribution = pd.DataFrame(
    {
        label: df_problem[column_name.replace("_num", "")].value_counts(normalize=True).reindex(
            ["Rarely", "Sometimes", "Often", "Always"], fill_value=0
        ).mul(100)
        for label, column_name in frequency_problem_mapping.items()
    }
).transpose().round(1)
display(frequency_distribution)

ax = frequency_distribution.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 5),
    color=["#b7d5e5", "#7fb3d5", "#3d7ea6", "#1f4e79"],
)
ax.set_title("How Often Participants Encounter These Frictions")
ax.set_ylabel("Participants (%)")
ax.set_xlabel("")
plt.xticks(rotation=20, ha="right")
plt.legend(title="Response")
plt.tight_layout()
plt.show()


The frequency-based items reveal a strong repeated-friction pattern. For all three measures, `77.8%` of participants report experiencing the issue **sometimes or more**. In other words, even where the average severity is moderate, the problems are recurring rather than isolated.

This is particularly useful for problem validation because it suggests that the experience breaks down during regular use. Visibility issues, unhelpful feedback, and periods of disengagement from the data appear to be part of the normal wearable journey rather than edge cases.

#### 3. Current Users vs Former or Non-Current Users

To better understand why people may stop using wearables, we compare current users with participants who said they do **not currently use** wearable technologies. This gives us a simple bivariate view of whether disengaged users report stronger pain points.

In [ ]:
df_problem["user_status"] = df_problem["usage_duration"].apply(
    lambda value: "Former / non-current users"
    if value == "I do not currently use them"
    else "Current users"
)

comparison_mapping = {
    "Information Overload": "information_overload",
    "Interpretation Difficulty": "data_interpretation_difficulty",
    "Frustration": "frustrated_feeling",
    "Battery Anxiety": "battery_anxiety",
    "Actionable Feedback Gap": "actionable_insight_gap",
    "Visibility Difficulties": "info_visibility_diff_num",
    "Feedback Not Useful": "feedback_not_useful_frequency_num",
    "Stopped Checking Data": "stop_checking_data_frequency_num",
}

group_comparison = (
    df_problem.groupby("user_status")[list(comparison_mapping.values())]
    .mean()
    .transpose()
    .rename(index={value: key for key, value in comparison_mapping.items()})
    .round(2)
)
display(group_comparison)

ax = group_comparison.plot(kind="bar", figsize=(11, 5), color=["#3d7ea6", "#d1495b"])
ax.set_title("Problem Indicators by Current Wearable Use Status")
ax.set_ylabel("Average response")
ax.set_xlabel("")
plt.xticks(rotation=25, ha="right")
plt.legend(title="Participant group")
plt.tight_layout()
plt.show()


Former or non-current users report higher averages for **information overload**, **interpretation difficulty**, **frustration**, **battery anxiety**, **visibility difficulties**, and **feedback not being useful**. This supports the idea that people may stop using wearables not because the concept lacks value, but because the experience becomes too effortful or insufficiently rewarding.

Interestingly, current users score higher on **stopping checking data** from time to time. This suggests disengagement can happen even among people who have not fully abandoned the technology, which points to an ongoing retention problem rather than a one-time dropout event.

#### Step 3 Reflection

Overall, the problem appears both **relevant** and **meaningful** for this sample. The strongest evidence is not that every metric is extremely high, but that multiple frictions appear repeatedly across the dataset. Battery anxiety and the gap between data and actionable feedback stand out as the strongest agreement-based concerns, while visibility issues, unhelpful feedback, and periods of disengagement are all commonly experienced over time.

This means the problem space is best described as a **moderate but persistent** set of wearable-use frustrations. For design, this points toward opportunities to reduce cognitive load, make feedback easier to act on, and better support long-term engagement rather than only first-time use.